# WebMD Preprocessing
Prepare WebMD for additional training and numerical aspect supervision.

In [6]:
# DEV MODE keeps notebook runs small during development; set to False for full DGX experiments.
DEV_MODE = True
SAMPLE_SIZE = 10000

# Resolve the project root from either the repository root or notebooks/ directory.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import the shared device and seed utilities required by the specification.
from utils.device import RANDOM_SEED, get_device, set_seed

set_seed(RANDOM_SEED)
DEVICE = get_device()


Selected device: CUDA (NVIDIA GeForce RTX 5070 Ti Laptop GPU)


In [7]:
import pandas as pd


# Shared preprocessing utilities used identically across datasets.
import html
import re

LABEL2ID = {"Negative": 0, "Neutral": 1, "Positive": 2}
ID2LABEL = {0: "Negative", 1: "Neutral", 2: "Positive"}
NUM_CLASSES = 3


def clean_text(value):
    # Apply the exact text cleaning pipeline from the specification.
    if not isinstance(value, str):
        return ""
    text = value.lower()
    text = html.unescape(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s'-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def label_from_rating_10(rating):
    # Map Drugs.com 1-10 ratings to Negative/Neutral/Positive labels.
    rating = float(rating)
    if 1 <= rating <= 4:
        return 0
    if 5 <= rating <= 6:
        return 1
    if 7 <= rating <= 10:
        return 2
    return None


def label_from_rating_5(rating):
    # Map 1-5 ratings to Negative/Neutral/Positive labels.
    rating = float(rating)
    if 1 <= rating <= 2:
        return 0
    if rating == 3:
        return 1
    if 4 <= rating <= 5:
        return 2
    return None


def label_from_effectiveness(value):
    # Map Druglib effectiveness text to the three-class label space.
    mapping = {
        "Ineffective": 0,
        "Marginally Effective": 1,
        "Moderately Effective": 1,
        "Considerably Effective": 2,
        "Highly Effective": 2,
    }
    return mapping.get(value, None)


def label_from_side_effects(value):
    # Map Druglib side-effect severity text to the three-class label space.
    mapping = {
        "Severe Side Effects": 0,
        "Extremely Severe Side Effects": 0,
        "Mild Side Effects": 1,
        "Moderate Side Effects": 1,
        "No Side Effects": 2,
    }
    return mapping.get(value, None)


## Load and Normalize Columns

In [8]:
raw_path = PROJECT_ROOT / "data/raw/webmd_raw.csv"
df = pd.read_csv(raw_path)
if DEV_MODE:
    df = df.head(SAMPLE_SIZE)
print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())

# The downloaded WebMD file uses capitalized names; normalize them to the specification.
rename_map = {
    "Reviews": "review",
    "Effectiveness": "effectiveness",
    "EaseofUse": "easeOfUse",
    "Satisfaction": "satisfaction",
    "Sides": "sideEffectsReview",
}
df = df.rename(columns=rename_map)
if "rating" not in df.columns:
    df["rating"] = df["satisfaction"]
keep_cols = ["review", "rating", "effectiveness", "easeOfUse", "satisfaction", "sideEffectsReview"]
df = df[keep_cols].dropna().drop_duplicates().copy()


Raw shape: (10000, 12)
Columns: ['Age', 'Condition', 'Date', 'Drug', 'DrugId', 'EaseofUse', 'Effectiveness', 'Reviews', 'Satisfaction', 'Sex', 'Sides', 'UsefulCount']


## Clean Text and Assign Labels

In [9]:
df["review"] = df["review"].apply(clean_text)
df["sideEffectsReview"] = df["sideEffectsReview"].apply(clean_text)
df = df[(df["review"].str.len() > 0) & (df["sideEffectsReview"].str.len() > 0)].copy()
df["label"] = df["rating"].apply(label_from_rating_5)
df["efficacy_label"] = df["effectiveness"].apply(label_from_rating_5)
df["ease_label"] = df["easeOfUse"].apply(label_from_rating_5)
df["satisfaction_label"] = df["satisfaction"].apply(label_from_rating_5)

# WebMD side-effect text is retained, but no supervised side-effect label is assigned here.
df["side_effects_label"] = -100
df = df.dropna(subset=["label", "efficacy_label", "ease_label", "satisfaction_label"]).copy()
for column in ["label", "efficacy_label", "ease_label", "satisfaction_label", "side_effects_label"]:
    df[column] = df[column].astype(int)


## Verify and Save

In [10]:
for column in ["label", "efficacy_label", "ease_label", "satisfaction_label"]:
    print(f"\n{column} distribution")
    print(df[column].value_counts(normalize=True).sort_index())

output_path = PROJECT_ROOT / "data/processed/webmd_clean.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print("Saved:", output_path)
print("Final shape:", df.shape)
display(df.head())



label distribution
label
0    0.433521
1    0.145117
2    0.421362
Name: proportion, dtype: float64

efficacy_label distribution
efficacy_label
0    0.268793
1    0.170088
2    0.561119
Name: proportion, dtype: float64

ease_label distribution
ease_label
0    0.152700
1    0.112694
2    0.734606
Name: proportion, dtype: float64

satisfaction_label distribution
satisfaction_label
0    0.433521
1    0.145117
2    0.421362
Name: proportion, dtype: float64
Saved: D:\33333\MedSentiX\data\processed\webmd_clean.csv
Final shape: (7649, 11)


,review,rating,effectiveness,easeOfUse,satisfaction,sideEffectsReview,label,efficacy_label,ease_label,satisfaction_label,side_effects_label
0,i'm a retired physician and of all the meds i ...,5,5,5,5,drowsiness dizziness dry mouth nose throat hea...,2,2,2,2,-100
1,cleared me right up even with my throat hurtin...,5,5,5,5,drowsiness dizziness dry mouth nose throat hea...,2,2,2,2,-100
6,haven't gotten pregnant so it does it's job i ...,2,5,5,2,nausea vomiting headache bloating breast tende...,0,2,2,0,-100
7,i have take this for 5 years age 45-50 to prev...,5,5,5,5,nausea vomiting headache bloating breast tende...,2,2,2,2,-100
9,the 12 hour spray only works for me for 6 hours,2,2,4,2,temporary burning stinging dryness in the nose...,0,0,2,0,-100
